# Memory Cache

> In-memory caching for scan results.

In [ ]:
#| default_exp cache.memory

In [ ]:
#| export
import time
from dataclasses import dataclass, field
from typing import List, Optional

from cjm_file_discovery.core.models import FileInfo

## ScanCache

In-memory cache for scan results with time-based expiration.

In [ ]:
#| export
@dataclass
class ScanCache:
    """In-memory cache for scan results with time-based expiration."""
    duration_seconds: int = 300  # Cache duration (default 5 minutes)
    _files: List[FileInfo] = field(default_factory=list, repr=False)
    _timestamp: Optional[float] = field(default=None, repr=False)

    def is_valid(self) -> bool:  # True if cache is valid and not expired
        """Check if cache is valid and not expired."""
        if self._timestamp is None:
            return False
        return (time.time() - self._timestamp) < self.duration_seconds

    def get(self) -> Optional[List[FileInfo]]:  # Cached files or None if invalid
        """Get cached files if valid."""
        if self.is_valid():
            return self._files
        return None

    def set(
        self,
        files: List[FileInfo]  # Files to cache
    ) -> None:
        """Update cache with new files."""
        self._files = files
        self._timestamp = time.time()

    def clear(self) -> None:
        """Clear the cache."""
        self._files = []
        self._timestamp = None

    @property
    def age_seconds(self) -> Optional[float]:  # Age in seconds or None if not set
        """Get cache age in seconds."""
        if self._timestamp is None:
            return None
        return time.time() - self._timestamp

    @property
    def file_count(self) -> int:  # Number of cached files
        """Get number of cached files."""
        return len(self._files)

In [ ]:
from cjm_file_discovery.core.models import FileType

# Test ScanCache
cache = ScanCache(duration_seconds=2)

# Initially invalid
assert cache.is_valid() == False
assert cache.get() is None
assert cache.file_count == 0

# Set some files
test_files = [
    FileInfo(name="test1.txt", path="/test1.txt", is_directory=False),
    FileInfo(name="test2.mp3", path="/test2.mp3", is_directory=False, file_type=FileType.AUDIO),
]
cache.set(test_files)

# Now valid
assert cache.is_valid() == True
assert cache.get() == test_files
assert cache.file_count == 2
assert cache.age_seconds is not None and cache.age_seconds < 1

# Clear
cache.clear()
assert cache.is_valid() == False
assert cache.get() is None

print("ScanCache tests passed!")

ScanCache tests passed!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()